# 8. Flows

A **flow** moves population between compartments, or into or out of the system.
You declare flows over selectors on a `PropertyMap`, compile them once to a
`CompiledModel`, and evaluate a JAX vector field. `euler` returns the **final
state only** — there is no trajectory or results frame yet.

This chapter uses `from summer4 import ...` throughout.

## A map, then a model

Build an SIR stratified by age, with severity only on the infectious. That
ragged axis is the same one {doc}`04-ragged-stratification` introduced.

In [ ]:
from summer4 import (
    Dest,
    FlowModel,
    Overwrite,
    Property,
    PropertyMap,
    Source,
    TraitChain,
    TransitionFlow,
    euler,
)

state = Property("state", ("S", "I", "R"))
age = Property("age", ("0-4", "5-9", "10+"))
severity = Property("severity", ("mild", "severe"))

pmap = (
    PropertyMap.from_property(state)
    .stratify(age)
    .stratify(severity, where=state["I"])
)

assert pmap.size == 12  # 3 ages × (S + 2 I + R)
model = FlowModel(pmap)

## Transition flows

Infection `S → I` fans out across the two severity strata (explicit `split=`).
Recovery `I → R` matches leftover age automatically. An `Overwrite` with
`where=` zeros infection in the youngest band.

In [ ]:
model.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        0.2,
        split={severity: {"mild": 0.7, "severe": 0.3}},
        adjust=[Overwrite(0.0, where=age["0-4"])],
    )
)
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 0.1))

## Ageing as one named flow

`TraitChain` is a pairing override, not a subclass: one flow lists every band
step. Leftover properties (here `state`, and `severity` on `I`) still match.

In [ ]:
model.add_flow(
    TransitionFlow(
        "ageing",
        age.present(),
        age.present(),
        0.2,
        pairing=TraitChain(age, (("0-4", "5-9"), ("5-9", "10+"))),
    )
)

## Compile and query edges

`compile()` actualizes every join once and returns a static `CompiledModel`.
`edges(name)` is a query API: wrap compartment selectors in `Source` / `Dest`.
Bare traits raise — polarity is required.

In [ ]:
compiled = model.compile()
infection = compiled.edges("infection")

assert infection.n_edges == 6  # 3 ages × 2 severity dests
labels = infection.labels()
assert "state=S_age=5-9 -> state=I_age=5-9_severity=mild" in labels
assert "state=S_age=0-4 -> state=I_age=0-4_severity=severe" in labels

young = infection.select(Source(age["0-4"]))
assert young.size == 2

mild_from_mid = infection.select(Source(age["5-9"]) & Dest(severity["mild"]))
assert mild_from_mid.size == 1

# Infection changes state, never age.
assert infection.moves_mask(state).all()
assert not infection.moves_mask(age).any()

try:
    infection.select(state["S"])
except TypeError as exc:
    assert "Source" in str(exc)

## Step the field

`vector_field(t, y, params)` is JAX. `euler` takes a fixed step and returns
the last state. Young susceptibles do not infect (the `Overwrite`); total mass
is conserved because every flow here is a transition.

In [ ]:
import numpy as np

y0 = np.zeros(pmap.size)
y0[pmap.select(state["S"])] = 100.0
y0[pmap.select(state["I"] & severity["mild"])] = 10.0

dy = np.asarray(compiled.vector_field(0.0, y0, {}))
assert np.isclose(dy.sum(), 0.0)
s_young = pmap.select_one(state["S"] & age["0-4"])
# Overwrite zeros infection; ageing still moves 0-4 → 5-9.
assert np.isclose(dy[s_young], -0.2 * y0[s_young])

y1 = np.asarray(euler(compiled.vector_field, 0.0, y0, {}, dt=0.5, steps=4))
assert np.isclose(y1.sum(), y0.sum())
assert (y1[pmap.select(state["R"])] > 0).any()

Entry and exit flows (`EntryFlow`, `ExitFlow`) use the same compile path;
replacement births read another flow's mass with `death.sum_over(location)`.
See the API reference and {doc}`07-from-summer2` for the summer2 names these
map onto.